[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB02_image_processing.ipynb)

# NB02 · Image processing: from scratch to libraries
**Artificial Intelligence in Medical Imaging** · Day 1 · Lab 2 (40 min)

| | |
|---|---|
| Runtime | CPU |
| Data | the synthetic CT slice from NB01 (copied below, so this notebook stands alone) |
| Checkpoints | your convolution equals SciPy's · bone mask size · cleaned mask has one lung pair |
| Tiers | **Core** = TODO cells · **Plus** = choose a filter and justify · **Challenge** = separable Gaussian and speed |

**Learning goals.** (1) Window and threshold an image. (2) Write a 2D convolution and check it against a library. (3) Use Gaussian, median, CLAHE, Sobel, Otsu and morphology, and say what problem each one solves. (4) Notice that every preprocessing step changes what a model can learn.

In [ ]:
import time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # scikit-image renames some arguments between versions
import numpy as np, matplotlib.pyplot as plt
from scipy import ndimage as ndi
import cv2
from skimage import filters, morphology, exposure, measure, segmentation
print("OpenCV", cv2.__version__)

In [ ]:
def make_ct_phantom(n=256, seed=0):
    """Synthetic chest-like CT slice in Hounsfield units (HU). Not patient data.
    Returns hu (float32) and a dict of masks. A low-contrast lesion (+35 HU) hides in soft tissue,
    and a lung nodule hides in the lung."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:n, 0:n].astype(np.float32)
    cy, cx = n / 2, n / 2
    hu = np.full((n, n), -1000.0, np.float32)                       # air
    body = ((xx - cx) / (0.42 * n)) ** 2 + ((yy - cy) / (0.32 * n)) ** 2 <= 1
    hu[body] = 40.0                                                  # soft tissue
    lungL = ((xx - (cx - 0.17 * n)) / (0.12 * n)) ** 2 + ((yy - cy) / (0.20 * n)) ** 2 <= 1
    lungR = ((xx - (cx + 0.17 * n)) / (0.12 * n)) ** 2 + ((yy - cy) / (0.20 * n)) ** 2 <= 1
    lungs = lungL | lungR
    hu[lungs] = -800.0
    spine = ((xx - cx) ** 2 + (yy - (cy + 0.22 * n)) ** 2) <= (0.04 * n) ** 2
    hu[spine] = 450.0                                                # bone
    lesion = ((xx - cx) ** 2 + (yy - (cy - 0.02 * n)) ** 2) <= (0.035 * n) ** 2
    hu[lesion] = 75.0                                                # soft tissue + 35 HU
    nodule = ((xx - (cx - 0.17 * n)) ** 2 + (yy - (cy - 0.05 * n)) ** 2) <= (0.02 * n) ** 2
    hu[nodule] = -450.0                                              # nodule inside lung
    water = ((xx - (cx + 0.05 * n)) ** 2 + (yy - (cy + 0.12 * n)) ** 2) <= (0.025 * n) ** 2
    hu[water] = 0.0                                                  # water-like cyst
    hu += rng.normal(0, 8, hu.shape).astype(np.float32)             # scanner noise
    return hu, {"body": body, "lungs": lungs, "lesion": lesion, "nodule": nodule, "spine": spine, "water": water}

def apply_window(hu, level, width):
    lo, hi = level - width / 2, level + width / 2
    return np.round((np.clip(hu, lo, hi) - lo) / (hi - lo) * 255).astype(np.uint8)

hu, masks = make_ct_phantom(n=128, seed=0)       # 128 x 128 keeps the from-scratch loops fast
img8 = apply_window(hu, 40, 400)                  # soft-tissue window
plt.figure(figsize=(4, 4)); plt.imshow(img8, cmap="gray"); plt.axis("off"); plt.title("soft-tissue window"); plt.show()

## 1. Predict, then run (Sobel)
In class we vote on what this kernel does:
```
-1 0 1
-2 0 2
-1 0 1
```
**Your prediction (write it down):** blur / vertical edges / horizontal edges / brighten. We check it in section 4.

## 2. Point operations: windowing and thresholding
**TODO 1.** Make a binary mask of bone: voxels above 200 HU in the original `hu` image.

In [ ]:
# TODO 1: boolean mask of pixels above 200 HU
bone_mask = None

In [ ]:
n_bone = int(bone_mask.sum())
assert bone_mask.dtype == bool and bone_mask.shape == hu.shape, "mask must be boolean, same shape as the image"
assert 50 < n_bone < 150, f"expected the spine only (about 80 pixels), got {n_bone}"
print("Checkpoint passed: spine found,", n_bone, "pixels")

## 3. Neighbourhood operations: convolution from scratch

Slide a kernel over the image: at each pixel multiply the neighbourhood by the kernel and add. We use zero padding so the output has the same size.

> *Note.* Deep-learning libraries (and this function) do **correlation**: the kernel is not flipped. For learned kernels it makes no difference.

**TODO 2.** Fill in `conv2d`.

In [ ]:
def conv2d(image, kernel):
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    padded = np.pad(image.astype(np.float64), ((ph, ph), (pw, pw)), mode="constant")
    out = np.zeros(image.shape, dtype=np.float64)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            # TODO 2: take the patch padded[i:i+kh, j:j+kw], multiply by the kernel, sum, store in out[i, j]
            pass
    return out

### Checkpoint 2 · your convolution equals SciPy's

In [ ]:
k_test = np.arange(9, dtype=float).reshape(3, 3)          # asymmetric on purpose
mine = conv2d(img8, k_test)
ref = ndi.correlate(img8.astype(np.float64), k_test, mode="constant")
assert np.allclose(mine, ref), "conv2d differs from scipy.ndimage.correlate"
print("Checkpoint 2 passed: max difference", np.abs(mine - ref).max())

### Blur: box and Gaussian
**TODO 3.** Build a Gaussian kernel of size `size` and standard deviation `sigma`, normalised so that it sums to 1 (why must it?).

In [ ]:
def gaussian_kernel(size, sigma):
    ax = np.arange(size) - (size - 1) / 2
    xx, yy = np.meshgrid(ax, ax)
    # TODO 3a: k = exp(-(x^2 + y^2) / (2 sigma^2))
    k = None
    # TODO 3b: normalise so that the kernel sums to 1
    return k

In [ ]:
g = gaussian_kernel(5, 1.2)
assert g is not None and abs(g.sum() - 1) < 1e-9 and g.shape == (5, 5) and g[2, 2] == g.max()
noisy = img8.astype(float) + np.random.default_rng(1).normal(0, 25, img8.shape)
box = conv2d(noisy, np.ones((5, 5)) / 25)
gau = conv2d(noisy, g)
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, im, t in zip(ax, [noisy, box, gau], ["noisy", "box 5x5", "Gaussian 5x5"]):
    a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")
plt.show()

### Edges: Sobel
**TODO 4.** Apply the two Sobel kernels with `conv2d` and combine them into an edge magnitude. Then check your prediction from section 1.

In [ ]:
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
sobel_y = sobel_x.T
# TODO 4: gx and gy with conv2d; magnitude = sqrt(gx^2 + gy^2)
gx = gy = mag = None
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, im, t in zip(ax, [gx, gy, mag], ["gx (sobel_x)", "gy", "magnitude"]):
    a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")
plt.show()

In [ ]:
# left body wall (a vertical edge) is near column 10; top body wall (a horizontal edge) is near row 23
assert mag is not None
gx_side, gy_side = np.abs(gx[60:68, 7:14]).mean(), np.abs(gy[60:68, 7:14]).mean()
gx_top, gy_top = np.abs(gx[20:27, 60:68]).mean(), np.abs(gy[20:27, 60:68]).mean()
print(f"left wall: gx {gx_side:.0f}, gy {gy_side:.0f} | top wall: gx {gx_top:.0f}, gy {gy_top:.0f}")
assert gx_side > 3 * gy_side and gy_top > 3 * gx_top, "sobel_x should find vertical edges, sobel_y horizontal ones"
print("Checkpoint passed. Answer to section 1: vertical edges (the kernel measures change along x).")

> **Pitfall: 8-bit overflow.** Adding two `uint8` images wraps around at 255. Always convert to float before arithmetic.

In [ ]:
a = np.array([200], np.uint8); b = np.array([100], np.uint8)
print("uint8 :", a + b, " <- wrong (300 mod 256)")
print("float :", a.astype(float) + b)
assert (a + b)[0] == 44

## 4. The same tools from libraries
A one-line library call is faster and tested, but you should know what it does. Keep this table in mind: *choose the tool from the problem, not the other way round.*

| Problem | Tool | Watch out for |
|---|---|---|
| Noisy image | Gaussian or median | blur removes small lesions |
| Low contrast | windowing, CLAHE | CLAHE amplifies noise |
| Find boundaries | Sobel | sensitive to noise: smooth first |
| Bright vs dark | Otsu threshold | one global threshold rarely fits all images |

**TODO 5.** Use OpenCV Gaussian blur (σ chosen by OpenCV from the kernel size), the median filter, and CLAHE from scikit-image (`clip_limit=0.03`).

In [ ]:
noisy8 = np.clip(noisy, 0, 255).astype(np.uint8)
# TODO 5a: Gaussian blur with OpenCV: cv2.GaussianBlur(image, (5, 5), 0)
gauss_lib = None
# TODO 5b: median filter with OpenCV: cv2.medianBlur(image, 5)
median_lib = None
# TODO 5c: CLAHE: exposure.equalize_adapthist(img8, clip_limit=0.03)
clahe = None
edges_lib = filters.sobel(img8 / 255.0)

In [ ]:
assert gauss_lib is not None and median_lib is not None and clahe is not None
fig, ax = plt.subplots(1, 5, figsize=(18, 4))
for a, im, t in zip(ax, [noisy8, gauss_lib, median_lib, clahe, edges_lib], ["noisy", "Gaussian", "median", "CLAHE", "Sobel (skimage)"]):
    a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")
plt.show()
# error against the clean image (lower = better denoising)
clean = img8.astype(float)
for name, im in [("noisy", noisy8), ("gaussian", gauss_lib), ("median", median_lib)]:
    print(f"{name:9s} mean abs error vs clean: {np.abs(im.astype(float) - clean).mean():.1f}")
assert np.abs(gauss_lib.astype(float) - clean).mean() < np.abs(noisy8.astype(float) - clean).mean()

### Otsu threshold and cleaning a mask
**TODO 6.** Segment the *air inside the lungs and outside the body* with Otsu on the noisy 8-bit image, then clean the mask: remove small objects and fill small holes (`morphology.remove_small_objects`, `ndi.binary_fill_holes`). Count the connected components before and after (expect many specks before, two lungs after).

In [ ]:
t = filters.threshold_otsu(noisy8)      # we segment the NOISY image on purpose
# TODO 6a: dark = pixels at or below the threshold
dark = None
raw_n = measure.label(dark).max()
# TODO 6b: remove objects smaller than 30 pixels (morphology.remove_small_objects),
#          drop objects touching the border (segmentation.clear_border), then fill holes (ndi.binary_fill_holes)
clean = None
clean_n = measure.label(clean).max()
print(f"Otsu threshold = {t:.0f} | components: raw {raw_n} -> cleaned {clean_n}")

In [ ]:
assert clean_n <= raw_n
# expected: the two lungs (the outside air was removed with clear_border)
assert clean_n == 2, f"expected 2 components (the two lungs), got {clean_n}"
lung_overlap = (clean & masks['lungs']).sum() / masks['lungs'].sum()
print(f"fraction of true lung pixels found: {lung_overlap:.2f}")
assert lung_overlap > 0.95
print("Checkpoint passed")

> **Reflection.** (1) The threshold found the lungs because the phantom is clean. On a real chest X-ray lung, heart and diaphragm overlap in brightness. What goes wrong? (2) Name one situation where Gaussian blur would hide something a clinician needs. (3) CLAHE made the structures clearer: did it add information?

> **Plus · choose a filter and justify.** Add salt-and-pepper noise (set 3% of pixels to 0 or 255). Compare Gaussian and median. Which preserves the body edge better? Support your answer with a number (mean absolute error on the edge band).

In [ ]:
# [PLUS]
# TODO (Plus): compare cv2.GaussianBlur and cv2.medianBlur on `sp`; report the mean abs error inside `band`
rng = np.random.default_rng(2)
sp = img8.copy(); r = rng.random(img8.shape)
sp[r < 0.015] = 0; sp[r > 0.985] = 255
band = ndi.binary_dilation(masks['body'], iterations=2) & ~ndi.binary_erosion(masks['body'], iterations=2)
...

> **Challenge · separable Gaussian.** A 2D Gaussian equals a 1D Gaussian along rows followed by one along columns. Show that the result is the same and time both on a 512 × 512 image.

In [ ]:
# [CHALLENGE]
big = np.random.default_rng(3).random((512, 512))
g1 = gaussian_kernel(9, 2.0)
# TODO (Challenge): build the 1D kernel k1 (same sigma, sums to 1);
# apply ndi.correlate1d along axis 0 then axis 1; compare with ndi.correlate(big, g1) and time both
...

## Summary
- Every filter is a convolution with a chosen kernel. A CNN learns the kernel.
- Check your own implementation against a library (checkpoint 2) before you trust either.
- Preprocessing is not neutral: it decides what the next step can see.

*Next:* NB03a (PyTorch taster).